## Creating a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

In [1]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [2]:
# Initialize and constants

load_dotenv(override=True)
gemini_api_key = os.getenv('GOOGLE_API_KEY')

if gemini_api_key and gemini_api_key.startswith('AQ.') and len(gemini_api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting!")
    
MODEL = 'gemini-3.5-flash-lite'

GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"

gemini = OpenAI(base_url=GEMINI_BASE_URL, api_key=gemini_api_key)

API key looks good so far


In [3]:
links = fetch_website_links("https://bakame03.github.io/portfolioX/")
links

['#hero',
 '#about',
 '#skills',
 '#resume',
 '#projects',
 '#services',
 '#contact',
 'https://x.com/bakame03',
 'https://www.instagram.com/bakame03/',
 'https://wa.me/+33758806942',
 'https://www.linkedin.com/in/aldo-alex-nganji-550072383',
 'mailto:nganjialdoalex@gmail.com',
 'https://github.com/Bakame03',
 '#projects',
 '#contact',
 'https://asyst.io',
 'https://bakame03.github.io/portfolioX/',
 'https://github.com/Bakame03',
 'https://github.com/Bakame03',
 'assets/img/portfolio/cv.pdf',
 'assets/img/portfolio/web/hypoxia.webp',
 'https://nebula-gray-seven.vercel.app/',
 'https://github.com/Bakame03/Nebula_DevArt_2026_Project_-HYPOXIA-',
 'assets/img/portfolio/web/eatwell.webp',
 'https://imaginative-cobbler-efa68f.netlify.app/',
 'https://github.com/Bakame03/Eat_Well',
 'assets/img/portfolio/web/services.webp',
 'https://celadon-frangollo-6483f9.netlify.app/',
 'https://github.com/Bakame03/service_web_project',
 'assets/img/portfolio/web/budget.webp',
 'https://budget-iu8r.onrend

## First step: Have Gemini figure out which links are relevant

### Use a call to it to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

In [4]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [6]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [7]:
print(get_links_user_prompt("https://bakame03.github.io/portfolioX/"))


Here is the list of links on the website https://bakame03.github.io/portfolioX/ -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

#hero
#about
#skills
#resume
#projects
#services
#contact
https://x.com/bakame03
https://www.instagram.com/bakame03/
https://wa.me/+33758806942
https://www.linkedin.com/in/aldo-alex-nganji-550072383
mailto:nganjialdoalex@gmail.com
https://github.com/Bakame03
#projects
#contact
https://asyst.io
https://bakame03.github.io/portfolioX/
https://github.com/Bakame03
https://github.com/Bakame03
assets/img/portfolio/cv.pdf
assets/img/portfolio/web/hypoxia.webp
https://nebula-gray-seven.vercel.app/
https://github.com/Bakame03/Nebula_DevArt_2026_Project_-HYPOXIA-
assets/img/portfolio/web/eatwell.webp
https://imaginative-cobbler-efa68f.netlify.app/
https://github.com/Bakame03/Eat_Well
asset

In [8]:
def select_relevant_links(url):
    response = gemini.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [9]:
select_relevant_links("https://bakame03.github.io/portfolioX/")

{'links': [{'type': 'about page',
   'url': 'https://bakame03.github.io/portfolioX/#about'},
  {'type': 'portfolio page',
   'url': 'https://bakame03.github.io/portfolioX/#projects'},
  {'type': 'services page',
   'url': 'https://bakame03.github.io/portfolioX/#services'},
  {'type': 'resume',
   'url': 'https://bakame03.github.io/portfolioX/assets/img/portfolio/cv.pdf'},
  {'type': 'social profile',
   'url': 'https://www.linkedin.com/in/aldo-alex-nganji-550072383'},
  {'type': 'social profile', 'url': 'https://github.com/Bakame03'}]}

In [10]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = gemini.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [11]:
select_relevant_links("https://bakame03.github.io/portfolioX/")

Selecting relevant links for https://bakame03.github.io/portfolioX/ by calling gemini-3.5-flash-lite
Found 7 relevant links


{'links': [{'type': 'about page',
   'url': 'https://bakame03.github.io/portfolioX/#about'},
  {'type': 'resume page',
   'url': 'https://bakame03.github.io/portfolioX/assets/img/portfolio/cv.pdf'},
  {'type': 'projects page',
   'url': 'https://bakame03.github.io/portfolioX/#projects'},
  {'type': 'services page',
   'url': 'https://bakame03.github.io/portfolioX/#services'},
  {'type': 'contact page',
   'url': 'https://bakame03.github.io/portfolioX/#contact'},
  {'type': 'social media',
   'url': 'https://www.linkedin.com/in/aldo-alex-nganji-550072383'},
  {'type': 'portfolio repository', 'url': 'https://github.com/Bakame03'}]}

In [12]:
select_relevant_links("https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gemini-3.5-flash-lite
Found 6 relevant links


{'links': [{'type': 'company page', 'url': 'https://huggingface.co/'},
  {'type': 'pricing page', 'url': 'https://huggingface.co/pricing'},
  {'type': 'enterprise page', 'url': 'https://huggingface.co/enterprise'},
  {'type': 'blog page', 'url': 'https://huggingface.co/blog'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'brand assets page', 'url': 'https://huggingface.co/brand'}]}

## Second step: make the brochure

Assemble all the details into another prompt to Gemini

In [13]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [14]:
print(fetch_page_and_all_relevant_links("https://bakame03.github.io/portfolioX/"))

Selecting relevant links for https://bakame03.github.io/portfolioX/ by calling gemini-3.5-flash-lite
Found 7 relevant links


Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.


## Landing Page:

Aldo Alex NGANJI - Back End Developer & AI Enthusiast

Skip to main content
EN
About
Skills
Resume
Projects
Services
Contact
Aldo Alex NGANJI
I'm
View my projects
Get in touch
About
I am passionate about building robust and scalable backend systems,
            and I am now expanding my horizons into Artificial Intelligence.
Back End Developer | Python & NestJS | AI Enthusiast
Passionate about building robust and scalable backend systems. Currently studying Computer Science at
Aix-Marseille University
, 
              and a graduate of Université du Lac Tanganyika in Software Engineering.
              Recently contributed as a Back End Developer at
Asyst Resources LTD
, 
              working with NestJS, Python, and modern backend architectures. 
              Now expanding into
Artificial Intelligence
to build smarter, more impactful systems.
Website:
My Website
City:
Arles, France
Languages:
English, French
Degree:
Bachelor
Email:
nganjialdoalex@gmail.com
Availabl

In [15]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""


In [16]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [17]:
get_brochure_user_prompt("Alex_Portfolio", "https://bakame03.github.io/portfolioX/")

Selecting relevant links for https://bakame03.github.io/portfolioX/ by calling gemini-3.5-flash-lite
Found 7 relevant links


Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.


"\nYou are looking at a company called: Alex_Portfolio\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nAldo Alex NGANJI - Back End Developer & AI Enthusiast\n\nSkip to main content\nEN\nAbout\nSkills\nResume\nProjects\nServices\nContact\nAldo Alex NGANJI\nI'm\nView my projects\nGet in touch\nAbout\nI am passionate about building robust and scalable backend systems,\n            and I am now expanding my horizons into Artificial Intelligence.\nBack End Developer | Python & NestJS | AI Enthusiast\nPassionate about building robust and scalable backend systems. Currently studying Computer Science at\nAix-Marseille University\n, \n              and a graduate of Université du Lac Tanganyika in Software Engineering.\n              Recently contributed as a Back End Developer at\nAsyst Resources LTD\n, \n              working with NestJS, Python, and mo

In [18]:
def create_brochure(company_name, url):
    response = gemini.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [39]:
create_brochure("Alex's portfolio", "https://bakame03.github.io/portfolioX/")

Selecting relevant links for https://bakame03.github.io/portfolioX/ by calling gemini-3.5-flash-lite
Found 7 relevant links


Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.


# Aldo Alex Nganji: Backend Development & AI Innovation

Welcome to the professional portfolio and practice of Aldo Alex Nganji, a dedicated Back End Developer and AI Enthusiast based in Arles, France. Specializing in Python, NestJS, and modern backend architectures, Aldo is committed to building robust, scalable systems that create a genuine, positive impact on people's lives.

## About Aldo Alex Nganji

With a strong foundation in Software Engineering from the Université du Lac Tanganyika (Class of 2024) and ongoing Computer Science studies at Aix-Marseille University, Aldo brings both academic rigor and real-world industry experience to every project. 

His journey into tech began in 2020, evolving quickly from early curiosity into a professional passion for modern backend depth, including microservices and SaaS architectures. Having previously contributed as a Back End Developer at Asyst Resources LTD—transitioning successfully from an internship to a full developer role—Aldo is now expanding his expertise into Artificial Intelligence to deliver even smarter, more impactful solutions.

## Company Culture & Philosophy

At the core of Aldo's work is a simple yet powerful mission: build things that genuinely help the people around us. Believing that utility and human impact give life and technology their true meaning, the operational culture emphasizes:
* Continuous learning and adaptation (bridging traditional software engineering with cutting-edge AI)
* High standards for system scalability, robustness, and modern architecture
* Clear communication and multilingual capabilities (fluent in English and French)

## Services & Expertise

Prospective clients and partners can collaborate on a range of technical needs, including:
* Robust and Scalable Backend Systems (Python, NestJS)
* Microservices and SaaS Architecture Design
* Intelligent System Integrations leveraging Artificial Intelligence

## Opportunities & Collaboration

Aldo is actively looking to connect with prospective customers, collaborators, and forward-thinking organizations. He is currently available for:
* Freelance Projects
* Internships
* Apprenticeships

## Get in Touch

Ready to build something impactful together? 

* **Location:** Arles, France
* **Email:** nganjialdoalex@gmail.com
* **Languages:** English, French
* **Connect:** Explore active projects and GitHub activity online to see modern engineering in action.

## changing this so that the results stream back from OpenAI, with the familiar typewriter animation

In [19]:
def stream_brochure(company_name, url):
    stream = gemini.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [20]:
stream_brochure("Alex's portfolio", "https://bakame03.github.io/portfolioX/")

Selecting relevant links for https://bakame03.github.io/portfolioX/ by calling gemini-3.5-flash-lite
Found 6 relevant links


Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.


# Aldo Alex Nganji: Backend Development & AI Engineering

Welcome to the professional portfolio and practice of Aldo Alex Nganji, a dedicated Back End Developer and AI Enthusiast based in Arles, France. Specializing in Python, NestJS, and modern backend architectures, Aldo builds robust, scalable systems designed to make a genuine impact.

## Who We Are & Our Mission
Founded on a passion for problem-solving and a transition from medical aspirations to computer science, our core mission is simple: **to build things that genuinely help the people around us.** We believe that technology should serve a meaningful purpose, improving lives through efficient software and intelligent systems.

## Company Culture
Our culture thrives on curiosity, continuous learning, and adaptability. 
* **Driven by Passion:** Starting from scratch in 2020, our journey is fueled by a genuine love for coding and technological discovery.
* **Global Perspective:** Operating internationally with roots in software engineering education from Université du Lac Tanganyika and current advanced studies at Aix-Marseille University in France.
* **Bilingual Communication:** Fluent in both English and French, enabling seamless collaboration with diverse teams and international clients.

## Services & Expertise
We specialize in creating future-proof digital solutions for modern businesses:
* **Robust Backend Systems:** Architecting and maintaining high-performance backends using Python and NestJS.
* **Scalable Architecture:** Designing microservices and SaaS solutions that grow alongside your business needs.
* **Artificial Intelligence Integration:** Expanding horizons into AI to deliver smarter, data-driven, and impactful systems.

## Customers & Partners
We partner with forward-thinking organizations seeking technical excellence. Notably, we have collaborated with **Asyst Resources LTD**, transitioning from a successful internship into a vital role as a Back End Developer specializing in microservices and modern backend depth. 

## Careers & Opportunities
Are you looking to collaborate or bring fresh talent to your team? We are actively available for:
* **Freelance Projects**
* **Internships**
* **Apprenticeships**

Whether you are an investor looking to back purposeful tech initiatives, a prospective customer needing scalable backend solutions, or an organization searching for dedicated engineering talent, get in touch today.

## Get in Touch
* **Location:** Arles, France
* **Email:** nganjialdoalex@gmail.com
* **Languages:** English, French

In [21]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("Alex's portfolio", "https://bakame03.github.io/portfolioX/")

Selecting relevant links for https://bakame03.github.io/portfolioX/ by calling gemini-3.5-flash-lite
Found 7 relevant links


Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.


# Alex's Portfolio & Development Services

Welcome to the professional portfolio of Aldo Alex Nganji, a dedicated Back End Developer and Artificial Intelligence Enthusiast based in Arles, France. Specializing in robust, scalable systems and modern backend architectures, Alex combines strong engineering fundamentals with a passion for creating technology that genuinely helps people.

## Company Culture & Core Values

Driven by a simple yet powerful mission—to build things that genuinely help those around us—the culture here is rooted in continuous learning, curiosity, and impact. From an unexpected beginning in 2020 that transformed a scientific background into a deep love for software engineering, the ethos centers on dedication, adaptability, and excellence. Operating globally with fluency in both English and French, we value precision, modern technological depth, and purposeful innovation.

## Technical Expertise & Services

We specialize in crafting high-performance digital infrastructure, with core competencies including:
- **Backend Development:** Building scalable backend systems, microservices, and SaaS architectures using Python and NestJS.
- **Artificial Intelligence:** Expanding horizons into AI to engineer smarter, more impactful systems.
- **Services Offered:** Available for freelance projects, internships, and apprenticeships to bring technical rigor to your team.

## Clients & Collaboration

Having contributed professionally as a Back End Developer at Asyst Resources LTD—transitioning successfully from an intern to a core developer working on cutting-edge backend architectures—we understand the fast-paced needs of modern tech companies and startups. We are eager to collaborate with forward-thinking prospective customers and partners looking to scale their technical capabilities.

## Careers & Opportunities

Are you an investor, recruiter, or organization looking for a passionate developer who combines academic grounding from Université du Lac Tanganyika and Aix-Marseille University with hands-on industry experience? Aldo Alex Nganji is currently open to new professional collaborations, internships, apprenticeships, and freelance engagements. 

## Get in Touch

Ready to build something impactful together? 
- **Location:** Arles, France
- **Email:** nganjialdoalex@gmail.com
- **Languages:** English, French